In [5]:
for t in ["Freedocument", "karte"]:
    print(t, [r[1] for r in conn.execute(f'PRAGMA table_info("{t}")')])

Freedocument ['Patients_ID', 'Date', 'document_type', 'text_data', 'Study_ID']
karte ['record_no', 'visit_type', 'visit_datetime', 'flag_note', 'Patient_ID', 'Study_ID', 'facility', 'karte_text', 'end_time', 'home_care_guidance', 'family_note', 'temperature', 'blood_pressure', 'pulse', 'pulse_rhythm', 'spo2', 'spo2_note', 'respiratory_rate', 'resp_rhythm', 'height', 'weight', 'updated_at', 'nurse_name', 'nurse_id', 'visit_duration_min', 'source_file']


In [6]:
import sqlite3, re, math, unicodedata
import pandas as pd

DB = "/Users/muna/Hana_research/data/db/Hana_Research.db"
DECIMALS = 0            # 0=整数切り捨て, 1=小数第1位まで
LO, HI = 50, 220
OUT = "height_extracted.csv"

PAT = re.compile(
    r"(?<![A-Za-z])(?:ht|height|身長|h)\s*[:=]?\s*(\d{2,3}(?:\.\d+)?)(?!\d)",
    re.IGNORECASE,
)

def floor_d(x):
    f = 10 ** DECIMALS
    v = math.floor(x * f) / f
    return int(v) if DECIMALS == 0 else v

def extract(text):
    """最初のマッチのみ採用。範囲外はNone。戻り値: (値, マッチ原文)"""
    if not isinstance(text, str):
        return None
    m = PAT.search(unicodedata.normalize("NFKC", text))
    if not m:
        return None
    v = float(m.group(1))
    return (floor_d(v), m.group(0)) if LO <= v <= HI else None

def from_numeric(x):
    try:
        v = float(x)
    except (TypeError, ValueError):
        return None
    return (floor_d(v), str(x)) if LO <= v <= HI else None

def parse_karte_date(s):
    """'YYYY/M/DD(曜)00:00' -> datetime。曜日・時刻は無視"""
    s = s.astype(str).map(lambda x: unicodedata.normalize("NFKC", x))
    d = s.str.extract(r"(\d{4})/(\d{1,2})/(\d{1,2})")
    return pd.to_datetime(
        d[0] + "-" + d[1] + "-" + d[2], format="%Y-%m-%d", errors="coerce"
    )

conn = sqlite3.connect(DB)

free = pd.read_sql(
    "SELECT rowid AS rid, Patients_ID AS Patient_ID, Date, text_data FROM Freedocument",
    conn,
)
free["dt"] = pd.to_datetime(free["Date"], errors="coerce")

karte = pd.read_sql(
    "SELECT rowid AS rid, Patient_ID, visit_datetime, karte_text, height FROM karte",
    conn,
)
karte["dt"] = parse_karte_date(karte["visit_datetime"])

# 日付パース失敗の確認
print("日付パース失敗  Freedocument:", free["dt"].isna().sum(),
      " karte:", karte["dt"].isna().sum())

# 日付昇順（NaTは最後）、同日はrowid順
free  = free.sort_values(["Patient_ID", "dt", "rid"])
karte = karte.sort_values(["Patient_ID", "dt", "rid"])

def first_hits(df, col, source, fn=extract):
    hits = {}
    for pid, val, dt in zip(df["Patient_ID"], df[col], df["dt"]):
        if pid in hits:
            continue
        r = fn(val)
        if r:
            hits[pid] = (r[0], source, r[1], dt)
    return pd.DataFrame(
        [(p, *v) for p, v in hits.items()],
        columns=["Patient_ID", "height_value", "table_name", "source_text", "record_date"],
    )

parts = [
    first_hits(free,  "text_data",  "Freedocument.text_data"),
    first_hits(karte, "karte_text", "karte.karte_text"),
    first_hits(karte, "height",     "karte.height", fn=from_numeric),
]
long = pd.concat(parts, ignore_index=True)

# 採用フラグ：優先順位（Freedocument → karte_text → karte.height）の最上位を1
prio = {"Freedocument.text_data": 1, "karte.karte_text": 2, "karte.height": 3}
long["_p"] = long["table_name"].map(prio)
long = long.sort_values(["Patient_ID", "_p"])
long["adopted"] = (~long.duplicated("Patient_ID")).astype(int)

# 抽出できなかった患者も残す
all_ids = pd.concat([free["Patient_ID"], karte["Patient_ID"]]).drop_duplicates()
missing = all_ids[~all_ids.isin(long["Patient_ID"])]
miss_df = pd.DataFrame({
    "Patient_ID": missing, "height_value": pd.NA, "table_name": "not_found",
    "source_text": pd.NA, "record_date": pd.NaT, "adopted": 1,
})

out = pd.concat([long.drop(columns="_p"), miss_df], ignore_index=True)
out["record_date"] = pd.to_datetime(out["record_date"]).dt.strftime("%Y-%m-%d")
out = out.sort_values(["Patient_ID", "adopted"], ascending=[True, False])
out = out[["Patient_ID", "height_value", "table_name", "source_text", "record_date", "adopted"]]
out.to_csv(OUT, index=False, encoding="utf-8-sig")

# 確認用
print("総患者数:", len(all_ids))
print("抽出成功:", out.loc[out.adopted == 1, "height_value"].notna().sum())
print("未抽出  :", len(missing))
print(out.loc[out.adopted == 1, "table_name"].value_counts())

日付パース失敗  Freedocument: 3  karte: 0
総患者数: 18075
抽出成功: 1409
未抽出  : 16666
table_name
not_found                 16666
Freedocument.text_data     1024
karte.karte_text            385
Name: count, dtype: int64
